In [1]:
import numpy as np
import scipy as sp
import pandas as pd

import math, time
import matplotlib.pyplot as plt
from serial import Serial

from pico3204D import Picoscope3204D

In [2]:
def getFFT(voltage):
    an = []
    bn = []
    duration = 0.02
    sampleRate = 50000
    n = int(duration*sampleRate)
    FFT = sp.fft.rfft(voltage)
    FFTfreqs = sp.fft.rfftfreq(n, 1/sampleRate)
    
    for num in range(0, 50):
        an.append(2*np.real(FFT[num]) /n)
        bn.append(-2*np.imag(FFT[num]) /n)
    #print('FFT done')
    return (an, bn, FFT, FFTfreqs)
 

In [3]:
def genSinewave(freq, t, attenuation):
    signal = np.sin(2*np.pi*freq*t)/attenuation
    return signal

In [4]:
def genHarm(num, freq, t, attenuation, phi):
    signal = np.sin(2*np.pi*num*freq*t + phi)/attenuation
    return signal

In [6]:
def loadGen(signal):
    waveform = [int(signal[n]*32767) for n in range(0, len(signal))]
    
    for point in waveform:
        value = point.to_bytes(2, 'big', signed = 'True')
        serialData.write(value)

In [7]:
def zero():
    zeroe = 0
    for n in range(0, 1000):
        res = zeroe.to_bytes(2, 'big')
        serialData.write(res)

In [8]:
def calculate_thd(Fourier_series, freq, base_freq):
    numerator = 0
    index_point = int(freq/base_freq)
    for i in range (index_point + 1, len(Fourier_series)):
        numerator += (math.pow(np.abs(Fourier_series[i]), 2))

    thd = 100*math.sqrt(numerator)/(np.abs(Fourier_series[index_point]))

    return thd

In [ ]:
def init(freq, t, attenuation):
    sinewave = genSinewave(freq, t, attenuation)
    loadGen(sinewave)
    time.sleep(0.5)
    voltage = picoscope.read_data(max_samples=1000, sample_rate=2502).ch_B # read_data возвращает три столбца - 'time' : time_axis, 'ch_A' : adc2mVChAMax, 'ch_B' : adc2mVChBMax
    time.sleep(0.2)
    zero()
    return voltage

In [ ]:
fsamp = 50000
duration = 0.02
N = int(fsamp*duration)
t = np.linspace(0, duration, N)
freq = 50

#harm_attenuation = 50
base_freq = 50


In [ ]:
serialData = Serial('COM5', baudrate=115200)
zero() # обнуление генератора
picoscope = Picoscope3204D()
picoscope.initialize_ports(channelA_range=10, channelB_range=9)

In [9]:
def calculus(signal):
    an_out, bn_out, F_out, FFTfreqs_out = getFFT(signal)
    THD = calculate_thd(F_out, freq, base_freq)
    ratio_3 = np.abs(F_out[3])/np.abs(F_out[1])
    ratio_5 = np.abs(F_out[5])/np.abs(F_out[1])
    ratio_7 = np.abs(F_out[7])/np.abs(F_out[1])
    ratio_9 = np.abs(F_out[9])/np.abs(F_out[1])
    
    return THD, ratio_3, ratio_5, ratio_7, ratio_9


In [ ]:
########## start waveform
signal = init(freq, t, attenuation)
THD, ratio_3, ratio_5, ratio_7, ratio_9 = calculus(signal)
print('Initial harm content is ', ratio_3, ratio_5, ratio_7, ratio_9)
print('Initial THD is', THD)

In [10]:
'''

def cycle(num, phi, harm_attenuation, attenuation, freq, t):
    sinewave = genSinewave(freq, t, attenuation)
    harm_sign = genHarm(num, freq, t, harm_attenuation, phi)
    signal = sinewave + harm_sign
    loadGen(signal)
    time.sleep(0.5)
    voltage = picoscope.read_data(max_samples=1000, sample_rate=2502).ch_B # read_data возвращает три столбца - 'time' : time_axis, 'ch_A' : adc2mVChAMax, 'ch_B' : adc2mVChBMax
    time.sleep(0.3)
    zero()
    THD, ratio_3, ratio_5, ratio_7, ratio_9 = calculus(voltage)
    res.append({
            "harm_num": num,
            "THD": THD, 
            "ratio_3": ratio_3,
            "ratio_5": ratio_5,
            "ratio_7": ratio_7,
            "ratio_9": ratio_9, 
            "phase": phi,
            "attenuation": harm_attenuation
            })
    
'''

In [ ]:
def harm_run(sinewave, num, phi, harm_attenuation, freq, t):
    
    harm_sign = genHarm(num, freq, t, harm_attenuation, phi)
    signal = sinewave + harm_sign
    loadGen(signal)
    time.sleep(0.5)
    voltage = picoscope.read_data(max_samples=1000, sample_rate=2502).ch_B # read_data возвращает три столбца - 'time' : time_axis, 'ch_A' : adc2mVChAMax, 'ch_B' : adc2mVChBMax
    time.sleep(0.3)
    zero()
    THD, ratio_3, ratio_5, ratio_7, ratio_9 = calculus(voltage)
    return THD, ratio_3, ratio_5, ratio_7, ratio_9

In [ ]:
def find_min(freq, t, attenuation, harm_num, harm_attenuation):

    sinewave = genSinewave(freq, t, attenuation)

    res = []
    for phi in range(0, 65):
        THD, ratio_3, ratio_5, ratio_7, ratio_9 = harm_run(sinewave, harm_num, phi/10, harm_attenuation, freq, t)

        res.append({
                "harm_num": harm_num,
                "THD": THD, 
                "ratio_3": ratio_3,
                "ratio_5": ratio_5,
                "ratio_7": ratio_7,
                "ratio_9": ratio_9, 
                "phase": phi/10,
                "attenuation": harm_attenuation
                })

    data = pd.DataFrame(res)
    data.to_csv(f"harm_min_{time.strftime("%Y-%m-%d_%H-%M")}.csv")
    print('Data saved')

In [ ]:
attenuation = 10
harm_attenuation = 100
harm_number = 3
find_min(freq, t, attenuation, harm_number, harm_attenuation)

    